
<div style="background-color: #000000; padding: 20px; border-radius: 12px;">

<img src="images/logo.png" width="100">

# Protein Preparation and Molecular Docking

We will begin by downloading the target protein structure from the **Protein Data Bank (PDB)**, preparing the structure for molecular docking, and subsequently performing docking simulations using **AutoDock Vina**.

Before starting the docking workflow, we will first review the tools and libraries required to successfully complete the experiment. Many of these libraries were introduced in previous notebooks, so this workflow builds on the computational biology skills and techniques developed in earlier sessions.

For this notebook, we will introduce one additional library: **MDAnalysis**. MDAnalysis is a Python library primarily designed for the analysis of molecular dynamics (MD) simulations. However, it also provides powerful functionality for reading, writing, and manipulating a wide range of molecular structure and trajectory formats. Its selection language allows users to efficiently identify and isolate specific **atoms, residues, chains, ligands, and other components** of a molecular system.

## Libraries and Tools

| **Library/Tool**  | **Abbreviation** | **Purpose**                                                                                                                         |
| ----------------- | ---------------- | ----------------------------------------------------------------------------------------------------------------------------------- |
| `rcsbsearchapi`   | N/A              | Provides functions for searching the Protein Data Bank using the mmCIF dictionary and retrieving structural information.            |
| `requests`        | N/A              | Used to access web resources and interact with online databases and APIs.                                                           |
| `os`              | N/A              | Provides operating-system functions for managing files, directories, and file paths.                                                |
| `nglview`         | `nv`             | Provides interactive visualization and inspection of molecular structures within a Jupyter notebook.                                |
| `MDAnalysis`      | `mda`            | A molecular dynamics analysis library used for reading, writing, manipulating, and selecting molecular structures and trajectories. |
| **AutoDock Vina** | `vina`           | Performs molecular docking by predicting ligand binding poses and estimating binding affinity.                                      |

## Workflow Overview

Throughout this workflow, we will use these tools to:

1. **Search for and retrieve the target protein structure** from the Protein Data Bank.
2. **Inspect the downloaded protein structure** and identify relevant chains, ligands, cofactors, and structural components.
3. **Select and isolate the appropriate protein chain** for docking.
4. **Remove unnecessary molecules and structural components**, such as crystallographic water molecules or unrelated ligands, where appropriate.
5. **Prepare the receptor structure** by ensuring that the protein contains the required atoms and structural information for docking.
6. **Convert the receptor into the appropriate docking format**, including the preparation of the PDBQT file required by AutoDock Vina.
7. **Define the docking search space** by determining the appropriate grid-box center and dimensions.
8. **Prepare the ligand molecules** in a format compatible with AutoDock Vina.
9. **Perform molecular docking** using AutoDock Vina.
10. **Analyze and visualize the docking poses**, including binding orientation, interacting residues, and predicted binding affinity.

The overall workflow can therefore be summarized as:

**PDB Structure Retrieval → Protein Inspection → Protein Preparation → Receptor PDBQT Generation → Ligand Preparation → Grid Box Definition → AutoDock Vina Docking → Pose Analysis**

> **Important:** Protein preparation is a critical step in molecular docking. Errors such as incorrect chain selection, missing atoms, inappropriate protonation states, retained crystallographic molecules, or incorrect file formats can significantly affect docking results. Therefore, each preparation step should be inspected before proceeding to the next stage.

## Expected Outcome

At the end of this notebook, we should have a **properly prepared receptor**, **prepared ligand(s)**, a clearly defined **docking search space**, and a set of **AutoDock Vina docking poses** that can be further analyzed to determine the predicted interactions between the ligand and target protein.


In [1]:
import os
import requests
# PDB ID
pdb_id = "1J3J"
# Create directory for PDB files
protein_directory = "protein_structures"
os.makedirs(protein_directory, exist_ok=True)
# Construct the correct PDB download URL
url = f"https://files.rcsb.org/download/{pdb_id}.pdb"
# Download the structure
pdb_request = requests.get(url)
print("Status code:", pdb_request.status_code)

Status code: 200


In [2]:
with open(f"{protein_directory}/{pdb_id}.pdb", "w+") as f:
    f.write(pdb_request.text)

# Visualizing the Protein Structure

Before preparing the protein for docking, we will first inspect its **3D structure** using **MDAnalysis** and **NGLView**.

**MDAnalysis** allows us to load and analyze the PDB structure using its central `Universe` object. We can use it to select chains, residues, and atoms, as well as perform basic structural analysis.

**NGLView** provides an interactive 3D visualization of the protein directly in the Jupyter notebook.

**Workflow:**
**PDB → MDAnalysis → Structure Inspection → NGLView Visualization → Protein Preparation**


In [3]:
import MDAnalysis as mda

# Load into MDA universe
u = mda.Universe(f"{protein_directory}/{pdb_id}.pdb")
u

<Universe with 9961 atoms>

In [4]:
import MDAnalysis as mda

# Load into MDA universe
u = mda.Universe(f"{protein_directory}/{pdb_id}.pdb")
u

<Universe with 9961 atoms>

After loading the PDB structure, MDAnalysis creates a Universe, which represents our molecular system. In this case, the structure contains 9,494 atoms.

We can inspect the protein visually using NGLView, an interactive molecular visualization tool designed for web applications and Jupyter notebooks. It allows us to explore the protein structure in 3D and examine its different components.

## check the number of chains in protein

In [5]:
# Number of chains
chains = u.select_atoms("protein").segments

print(f"Number of chains: {len(chains)}")
print("Chains:")

for chain in chains:
    print(f"  {chain.segid}")

Number of chains: 4
Chains:
  A
  B
  C
  D


## List ligands and other non-standard molecules

In [6]:
# Select residues that are not protein, water, or common ions
ligands = u.select_atoms(
    "not protein and not resname HOH WAT SOL NA CL K MG CA"
).residues

print("Ligands / non-standard residues:")

for res in ligands:
    print(f"  {res.resname}  (Residue ID: {res.resid})")

Ligands / non-standard residues:
  CP6  (Residue ID: 609)
  NDP  (Residue ID: 610)
  CP6  (Residue ID: 709)
  NDP  (Residue ID: 710)
  UMP  (Residue ID: 611)
  UMP  (Residue ID: 711)


In [7]:
for res in ligands:
    print(
        f"{res.resname}: "
        f"{len(res.atoms)} atoms, "
        f"residue {res.resid}"
    )

CP6: 17 atoms, residue 609
NDP: 48 atoms, residue 610
CP6: 17 atoms, residue 709
NDP: 48 atoms, residue 710
UMP: 20 atoms, residue 611
UMP: 20 atoms, residue 711


In [25]:
# Select ligand TOP, residue 703
top_ligand = u.select_atoms("resname CP6 and resid 609")

# Check the selection
print(f"Selected atoms: {len(top_ligand)}")
print(f"Residue: {top_ligand.resnames[0]}")
print(f"Residue ID: {top_ligand.resids[0]}")

# Display the ligand
top_ligand

Selected atoms: 17
Residue: CP6
Residue ID: 609


<AtomGroup with 17 atoms>

In [26]:
lig = top_ligand.write("ligands/cp6.pdb")

/Users/biohack/miniconda3/envs/vina/lib/python3.12/site-packages/MDAnalysis/coordinates/PDB.py:1282: UserWarning: Found no information for attr: 'formalcharges' Using default value of '0'
  warnings.warn(


### Extract only chain A

In [27]:
chain_A = u.select_atoms("segid A")
chain_A.write("protein_structures/chain_A.pdb")

In [28]:
import nglview as nv

top = chain_A.select_atoms("resname CP6")
view = nv.show_mdanalysis(chain_A)
view.clear_representations()
view.add_cartoon(selection="protein", color="spectrum")
view.add_component(top)
view.add_ball_and_stick(component=1)
view.center()
view

NGLWidget()

### Check if Chain A was successfully extracted

In [29]:
print(chain_A)
print("Atoms:", len(chain_A))
print("Residues:", len(chain_A.residues))
print("Residue range:", chain_A.residues.resids.min(), "-",chain_A.residues.resids.max())

<AtomGroup [<Atom 1: N of type N of resname MET, resid 1 and segid A and altLoc >, <Atom 2: CA of type C of resname MET, resid 1 and segid A and altLoc >, <Atom 3: C of type C of resname MET, resid 1 and segid A and altLoc >, ..., <Atom 2025: O of type O of resname HOH, resid 1679 and segid A and altLoc >, <Atom 2026: O of type O of resname HOH, resid 1680 and segid A and altLoc >, <Atom 2027: O of type O of resname HOH, resid 1681 and segid A and altLoc >]>
Atoms: 2027
Residues: 348
Residue range: 1 - 1681


### get TOP ligand coordinates

In [30]:
top = chain_A.select_atoms("resname CP6 and resid 609")

for atom in top:
    x, y, z = atom.position
    print(f"{atom.name:>4s}  X={x:10.3f}  Y={y:10.3f}  Z={z:10.3f}")

  N1  X=    29.876  Y=     4.105  Z=    56.353
  C2  X=    29.343  Y=     5.048  Z=    55.473
  C3  X=    29.533  Y=     4.155  Z=    57.687
  C4  X=    28.530  Y=     5.144  Z=    58.182
  C5  X=    28.047  Y=     6.113  Z=    57.147
  N6  X=    28.414  Y=     6.029  Z=    55.837
  C7  X=    28.105  Y=     5.238  Z=    59.560
  C8  X=    28.710  Y=     6.088  Z=    60.501
  C9  X=    28.273  Y=     6.145  Z=    61.845
 C10  X=    27.182  Y=     5.306  Z=    62.294
 C11  X=    26.541  Y=     4.453  Z=    61.401
 C12  X=    26.993  Y=     4.419  Z=    60.047
 N13  X=    30.032  Y=     3.259  Z=    58.567
 N14  X=    29.710  Y=     4.979  Z=    54.121
 C15  X=    27.051  Y=     7.159  Z=    57.569
 C16  X=    25.678  Y=     7.184  Z=    56.935
 CL1  X=    26.781  Y=     5.456  Z=    63.900


In [31]:
c = top.center_of_mass()
print(f"x={c[0]:.3f}, y={c[1]:.3f}, z={c[2]:.3f}")

x=28.073, y=5.301, z=59.109


In [32]:
# Visualize GridBox
import nglview as nv
view = nv.show_mdanalysis(chain_A)
view.clear_representations()
view.add_cartoon(color="grey")
top = chain_A.select_atoms("resname CP6")
view.add_component(top)
view.add_ball_and_stick(component=1, radius=0.4)
# 20 Å box centered on TOP
x, y, z = top.center_of_mass()
s = 10
p = [(x+i*s, y+j*s, z+k*s) for i,j,k in
     [(-1,-1,-1),(1,-1,-1),(1,1,-1),(-1,1,-1),
      (-1,-1,1),(1,-1,1),(1,1,1),(-1,1,1)]]
for a,b in [(0,1),(1,2),(2,3),(3,0),(4,5),(5,6),(6,7),(7,4),
            (0,4),(1,5),(2,6),(3,7)]:
    view.shape.add_cylinder(p[a], p[b], [1,0,0], 0.2)
view.center()
view

NGLWidget()

In [33]:
#Delete all ligands
'''
NDP: 48 atoms, residue 701
UMP: 20 atoms, residue 702
TOP: 21 atoms, residue 703
GOL: 6 atoms, residue 704
NDP: 48 atoms, residue 701
UMP: 20 atoms, residue 702
TOP: 21 atoms, residue 703
GOL: 6 atoms, residue 704
'''
import MDAnalysis as mda

u = mda.Universe("protein_structures/chain_A.pdb")

# Keep only protein atoms
protein = u.select_atoms("protein")

# Save ligand-free protein
protein.write("protein_structures/chain_A_clean.pdb")

print(f"Saved {len(protein)} protein atoms")

/Users/biohack/miniconda3/envs/vina/lib/python3.12/site-packages/MDAnalysis/coordinates/PDB.py:1282: UserWarning: Found no information for attr: 'formalcharges' Using default value of '0'
  warnings.warn(


Saved 1837 protein atoms


### Fixing the Protein Structure

Now that we have isolated the protein, the next step is to prepare it for further analysis by ensuring that the structure is complete and chemically appropriate. This includes checking for missing atoms, resolving alternate or multiple occupancies, and adding the appropriate hydrogen atoms.

To accomplish this, we will use PDB2PQR, a specialized software package designed for preparing protein structures. PDB2PQR analyzes the input structure, identifies structural issues such as missing atoms and alternate conformations, and adds missing atoms where appropriate. It can also assign hydrogen atoms and generate a properly prepared structure suitable for subsequent computational analyses such as molecular docking and molecular dynamics simulations.

In [34]:
! pdb2pqr --pdb-output=protein_structures/chain_A_cleanH.pdb --pH=7.4 protein_structures/chain_A_clean.pdb protein_structures/chain_A_clean.pqr --whitespace

/Users/biohack/miniconda3/envs/vina/lib/python3.12/pty.py:95: DeprecationWarning: This process (pid=3089) is multi-threaded, use of forkpty() may lead to deadlocks in the child.
  pid, fd = os.forkpty()


INFO:PDB2PQR v3.6.2: biomolecular structure conversion software.
INFO:Please cite:  Jurrus E, et al.  Improvements to the APBS biomolecular solvation software suite.  Protein Sci 27 112-128 (2018).
INFO:Please cite:  Dolinsky TJ, et al.  PDB2PQR: expanding and upgrading automated preparation of biomolecular structures for molecular simulations. Nucleic Acids Res 35 W522-W525 (2007).
INFO:Checking and transforming input arguments.
INFO:Loading topology files.
INFO:Loading molecule: protein_structures/chain_A_clean.pdb
ERROR:Error parsing line: invalid literal for int() with base 10: ''
ERROR:<REMARK     2>
ERROR:Truncating remaining errors for record type:REMARK

ERROR:['REMARK']
INFO:Setting up molecule.
INFO:Created biomolecule object with 221 residues and 1837 atoms.
INFO:Setting termini states for biomolecule chains.
INFO:Loading forcefield.
INFO:Loading hydrogen topology definitions.
INFO:Attempting to repair 1 missing atoms in biomolecule.
INFO:Added atom OXT to residue ASN A 231 

### Saving a protein PDBQT File
The PDB2PQR program outputs two files, a PDB file and a PQR file. The PDB file is similar to PDB files we have worked with before, except that it contains hydrogens. The PQR file is another molecular file format that is similar to a PDB, but contains information about atomic radii and atomic charges.

For use with AutoDock Vina, we need our protein file to be in the "PDBQT" format. PDBQT is a specialized file format used by AutoDock Vina and other AutoDock tools. Like the PQR format, the PDBQT format can also contain partial charges. We will load our PQR file and use MDAnalysis to write a PDBQT file.

In [35]:
from rdkit import Chem
from meeko import MoleculePreparation

pdb_file = "protein_structures/chain_A_clean.pdb"

mol = Chem.MolFromPDBFile(
    pdb_file,
    sanitize=False,
    removeHs=False
)

print(mol)

In [40]:
import os
import MDAnalysis as mda

# Directory for PDBQT files
pdbqt_directory = "pdbqt"
os.makedirs(pdbqt_directory, exist_ok=True)

### Ligand Preparation

For preparing small-molecule ligands for molecular docking, several tools can be used, including MDAnalysis. In this workflow, however, we will use Meeko, a tool specifically designed for preparing small molecules for AutoDock Vina and related docking programs.

Meeko is particularly useful because it handles the conversion of ligand structures into the PDBQT format required by Vina and facilitates the visualization and analysis of docking results. Before using Meeko, ensure that the ligand structure already contains the appropriate hydrogen atoms.

We will use Meeko from the command line, similar to the way we used PDB2PQR for protein preparation. Although Meeko also provides a Python API, the command-line interface is straightforward and convenient for common tasks such as converting an SDF file to PDBQT.

In the following cell, we will run a Meeko command to convert the prepared ligand from the molecule_manipulation workflow into a PDBQT file ready for molecular docking with AutoDock Vina.

In [41]:
# modified Br
!mk_prepare_ligand.py \
-i modified_ligand/pyrimethamine_Br_optimized.sdf \
-o pdbqt/pyrimethamine_Br_optimized.pdbqt

Input molecules processed: 1, skipped: 0
PDBQT files written: 1
PDBQT files not written due to error: 0
Input molecules with errors: 0


In [42]:
# prepare original pyrimethamine save it in pdbqt directory
!mk_prepare_ligand.py \
-i ligands/pyrimethamine.sdf \
-o pdbqt/pyrimethamine.pdbqt

Input molecules processed: 1, skipped: 0
PDBQT files written: 1
PDBQT files not written due to error: 0
Input molecules with errors: 0


In [43]:
## Check that the PDBQT was created
## If it returns True, your ligand is ready for the next AutoDock Vina docking step.
import os
print(os.path.exists("pdbqt/pyrimethamine_Br_optimized.pdbqt"))

True
